# End-to-End ML Pipeline on Tesla Sales and Production Data

## Objective

The goal of this project is to build an end-to-end machine learning pipeline using Tesla sales and production data from 2015–2025.

The project includes:

- Data preprocessing
- Exploratory Data Analysis (EDA)
- Feature engineering
- Regression modeling
- Hyperparameter tuning
- Time series forecasting
- Model evaluation

Target Variable:
- Estimated_Deliveries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

## Dataset Loading

The Tesla dataset contains information regarding:

- Deliveries
- Production Units
- Average Price
- Battery Capacity
- Vehicle Range
- CO₂ Savings
- Region
- Vehicle Model

The dataset covers Tesla's performance from 2015 to 2025.

In [ ]:
df = pd.read_csv("tesla_deliveries_dataset_2015_2025.csv")
print("Shape:", df.shape)
df.head()

In [ ]:
assert df.shape[0] > 0, "Dataset is empty"
assert df.shape[1] > 0, "No columns found"
print("Dataset loaded successfully.")

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

## Reflection

The dataset appears structured and contains both numerical and categorical variables.

The target variable for prediction will be Estimated_Deliveries. Features such as production units, battery capacity, average vehicle price, and region are expected to influence delivery numbers.

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(
    df["Estimated_Deliveries"],
    bins=30,
    kde=True
)
plt.title("Distribution of Estimated Deliveries")
plt.xlabel("Estimated Deliveries")
plt.ylabel("Frequency")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(
    data=df,
    x="Region",
    y="Estimated_Deliveries"
)
plt.title("Deliveries Across Regions")
plt.xlabel("Region")
plt.ylabel("Estimated Deliveries")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(
    data=df,
    x="Model",
    y="Estimated_Deliveries"
)
plt.title("Average Deliveries by Tesla Model")
plt.xlabel("Model")
plt.ylabel("Estimated Deliveries")
plt.show()

In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(
    df.select_dtypes(include=np.number).corr(),
    annot=True,
    cmap="coolwarm"
)
plt.title("Correlation Matrix")
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
yearly = (
    df.groupby("Year")["Estimated_Deliveries"]
      .mean()
      .reset_index()
)
sns.lineplot(
    data=yearly,
    x="Year",
    y="Estimated_Deliveries"
)
plt.title("Average Deliveries by Year")
plt.xlabel("Year")
plt.ylabel("Average Deliveries")

plt.show()

## EDA Findings

Several numerical variables exhibit positive correlation with deliveries.

Production units appear strongly associated with deliveries, which is expected because production capacity influences sales volume.

The visualizations also indicate differences in deliveries across regions and Tesla vehicle models.

In [ ]:
# To prevent target leakage, we sort chronologically and use lagged deliveries for the ratio
df = df.sort_values(["Year", "Month"]).reset_index(drop=True)
df["Lagged_Deliveries"] = df.groupby(["Region", "Model"])["Estimated_Deliveries"].shift(1)
df["Lagged_Deliveries"].fillna(df["Estimated_Deliveries"].median(), inplace=True)

df["Production_Delivery_Ratio"] = (
    df["Production_Units"] /
    df["Lagged_Deliveries"]
)
df.drop(columns=["Lagged_Deliveries"], inplace=True)

# Encode cyclical Month feature
df["Month_sin"] = np.sin(2 * np.pi * df["Month"] / 12.0)
df["Month_cos"] = np.cos(2 * np.pi * df["Month"] / 12.0)
df["Price_per_km"] = (
    df["Avg_Price_USD"] /
    df["Range_km"]
)
df.head()

In [ ]:
assert "Production_Delivery_Ratio" in df.columns
assert "Price_per_km" in df.columns
print("Feature engineering completed.")

In [ ]:
X = df.drop(
    columns=["Estimated_Deliveries"]
)
y = df["Estimated_Deliveries"]

In [ ]:
numerical_features = [
    "Year",
    "Month_sin",
    "Month_cos",
    "Production_Units",
    "Avg_Price_USD",
    "Battery_Capacity_kWh",
    "Range_km",
    "CO2_Saved_tons",
    "Production_Delivery_Ratio",
    "Price_per_km"
]
categorical_features = [
    "Region",
    "Model"
]

In [ ]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numerical_features),
    ("cat", categorical_transformer, categorical_features)
])

In [ ]:
# To prevent lookahead bias / temporal leakage, we split chronologically
split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
print(X_train.shape)
print(X_test.shape)

In [ ]:
lr_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])
lr_pipeline.fit(X_train, y_train)
lr_pred = lr_pipeline.predict(X_test)
lr_r2 = r2_score(y_test, lr_pred)
print("Linear Regression R²:", lr_r2)

In [ ]:
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        random_state=42
    ))
])

Hyperparameter Tuning

In [ ]:
param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [5, 10, None]
}
grid_search = GridSearchCV(
    rf_pipeline,
    param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

Best Parameters

In [ ]:
print(grid_search.best_params_)

Predictions

In [ ]:
best_model = grid_search.best_estimator_
predictions = best_model.predict(X_test)

In [ ]:
mae = mean_absolute_error(
    y_test,
    predictions
)
rmse = np.sqrt(
    mean_squared_error(
        y_test,
        predictions
    )
)
r2 = r2_score(
    y_test,
    predictions
)
print("MAE :", mae)
print("RMSE:", rmse)
print("R2  :", r2)

In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Random Forest"
    ],
    "R2 Score": [
        lr_r2,
        r2
    ]
})
comparison

In [ ]:
assert mae >= 0
assert rmse >= 0
assert -1 <= r2 <= 1
print("Evaluation completed successfully.")

## Model Performance

The Random Forest model achieved strong predictive performance on the delivery prediction task.

Hyperparameter tuning improved model robustness and helped identify the optimal model configuration.

The evaluation metrics indicate that the model successfully captures the relationship between production, pricing, vehicle specifications, and deliveries.

In [ ]:
df["Date"] = pd.to_datetime(
    df["Year"].astype(str) +
    "-" +
    df["Month"].astype(str)
)
monthly = (
    df.groupby("Date")["Estimated_Deliveries"]
    .sum()
    .reset_index()
)
monthly["Lag1"] = (
    monthly["Estimated_Deliveries"]
    .shift(1)
)
monthly["Lag2"] = (
    monthly["Estimated_Deliveries"]
    .shift(2)
)
monthly.dropna(inplace=True)

In [ ]:
train_size = int(
    len(monthly) * 0.8
)
train = monthly.iloc[:train_size]
test = monthly.iloc[train_size:]

In [ ]:
X_train_ts = train[
    ["Lag1", "Lag2"]
]
y_train_ts = train[
    "Estimated_Deliveries"
]
X_test_ts = test[
    ["Lag1", "Lag2"]
]
y_test_ts = test[
    "Estimated_Deliveries"
]
forecast_model = LinearRegression()
forecast_model.fit(
    X_train_ts,
    y_train_ts
)
forecast = forecast_model.predict(
    X_test_ts
)

In [ ]:
forecast_rmse = np.sqrt(
    mean_squared_error(
        y_test_ts,
        forecast
    )
)
print("Forecast RMSE:", forecast_rmse)

Forecast Plot

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(
    test["Date"],
    y_test_ts,
    label="Actual Deliveries"
)
plt.plot(
    test["Date"],
    forecast,
    label="Forecasted Deliveries"
)
plt.title("Time Series Forecasting")
plt.xlabel("Date")
plt.ylabel("Estimated Deliveries")
plt.legend()
plt.show()

## Forecasting Insights

The forecasting model was trained using lag-based features and a chronological train-test split.

The predicted delivery trend follows the general pattern of actual deliveries, demonstrating the usefulness of historical delivery values for future forecasting.

Forecast RMSE was used to quantify forecasting performance.

# Conclusion

An end-to-end machine learning workflow was successfully implemented using Tesla sales and production data.

The project demonstrated:

- Data preprocessing
- Exploratory Data Analysis
- Feature engineering
- Regression modeling
- Hyperparameter tuning
- Time series forecasting

The final solution provides meaningful insights into Tesla delivery patterns and showcases a complete machine learning pipeline following industry-standard practices.